In [ ]:
# Run this cell to set the working directory to the repo root.
from pathlib import Path
import os, sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "method.py").exists() and (p / "tools").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [1]:
import numpy as np
import pandas as pd

# Uses `Distribution`, `run_de`, and `fitness` in the file
from method import *

# Clean up the output
np.set_printoptions(precision=2, suppress=True)

# Random seed for reproducibility
SEED = 20241225

## Real data experiments in the paper
This file illustrates how to replicate the real data experiments, which is largely similar to the instruction given in `Readme.md`

Assume that the CGM data from Shah et al. (2019) and Brown et al. (2019) are accessed and processed as guided in the [Awesome-CGM](https://github.com/IrinaStatsLab/Awesome-CGM) repository, saved in `./data/shah2019_filtered.csv` and `./data/brown2019_filtered.csv`. These datasets are measured by *Dexcom G6*, having glucose ranges from 39 mg/dL to 401 mg/dL.

As the differential evolution (DE) is a stochastic algorithm, we fix `SEED=20241225` to reproduce the experimental results illustrated in the paper. 

*Device Information*:
- OS: Windows 11
- CPU: AMD Ryzen 7 8845HS
- `Python == 3.10.14`
- `numpy == 1.26.4`, `pandas == 2.2.2`, `scipy == 1.14.0`

In [2]:
# Load data
data_shah = pd.read_csv("./data/shah2019_filtered.csv")
grouped_data_shah = data_shah.groupby('id').agg({'gl': list}).reset_index()
data_class_shah = Distribution(grouped_data_shah["gl"], ran=(39., 401.), M=200)

data_brown = pd.read_csv("./data/brown2019_filtered.csv")
grouped_data_brown = data_brown.groupby('id').agg({'gl': list}).reset_index()
data_class_brown = Distribution(grouped_data_brown["gl"], ran=(39., 401.), M=200)

After making data into `Distribution` classes, we can run `run_de` with specified target number of thresholds `K` and the threshold-optimality criteria: `loss="Loss1"` or `loss="Loss2"`. 

If you want to see the optimization progress, set `disp=True` in the `run_de` function. The function typically takes a couple of minutes, depending on the device.

Experiments conducted in the paper are given below.

### Shah dataset

##### $L_1$ loss

In [3]:
# Selecting K=4 thresholds
best_cutoffs, min_loss = run_de(data_class_shah, K=4, loss="Loss1", seed=SEED)
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [ 75.83 100.69 123.7  154.96]
Obtained loss: 16.619439163631053


In [ ]:
# Selecting K=2 thresholds
best_cutoffs, min_loss = run_de(data_class_shah, K=2, loss="Loss1", seed=SEED)
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [ 71.69 127.91]
Obtained loss: 88.90773604650336


##### $L_2$ loss (supplementary)

In [4]:
# Selecting K=4 thresholds
best_cutoffs2, min_loss2 = run_de(data_class_shah, K=4, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 74.75 101.48 127.28 163.79]
Obtained loss: 1.1462606747371855


In [4]:
# Selecting K=2 thresholds
best_cutoffs2, min_loss2 = run_de(data_class_shah, K=2, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [119.62 163.88]
Obtained loss: 16.139003918338258


### Brown dataset

##### $L_1$ loss

In [ ]:
# Selecting K=4 thresholds
best_cutoffs, min_loss = run_de(data_class_brown, K=4, loss="Loss1", seed=SEED)
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [ 84.88 171.2  232.62 301.58]
Obtained loss: 41.215727144747326


In [ ]:
# Selecting K=2 thresholds
best_cutoffs, min_loss = run_de(data_class_brown, K=2, loss="Loss1", seed=SEED)
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [210.48 288.43]
Obtained loss: 398.2131637666346


##### Semi-supervised, fixing 70 and 181 mg/dL

In [ ]:
# Selecting K=2 additional thresholds (4 thresholds total)
best_cutoffs, min_loss = run_de(data_class_brown, K=2, loss="Loss1", seed=SEED, fixed=(70, 181))
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [ 70.   181.   240.9  305.95]
Obtained loss: 64.05054061137514


##### $L_2$ loss (supplementary)

In [5]:
# Selecting K=4 thresholds
best_cutoffs2, min_loss2 = run_de(data_class_brown, K=4, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 91.08 172.82 269.23 400.64]
Obtained loss: 1.6870185244975084


In [6]:
# Selecting K=2 thresholds
best_cutoffs2, min_loss2 = run_de(data_class_brown, K=2, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [182.37 280.41]
Obtained loss: 9.472764134773206


In [7]:
# Selecting 2 thresholds while fixing 70
best_cutoffs2, min_loss2 = run_de(data_class_brown, K=1, loss="Loss2", seed=SEED, fixed=[70.])
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 70.   221.24]
Obtained loss: 28.750177071633964


### Combined data

In [3]:
data_concat = pd.concat([grouped_data_shah, grouped_data_brown], ignore_index=True)
data_class_concat = Distribution(data_concat["gl"], ran=(39., 401.), M=200)

##### $L_2$ loss

Selecting $K=2$ thresholds

In [5]:
# Selecting K=2 thresholds
best_cutoffs2, min_loss2 = run_de(data_class_concat, K=2, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [149.7  257.25]
Obtained loss: 22.163945849800697


In [4]:
# Selecting K=2 thresholds with W_1
best_cutoffs2, min_loss2 = run_de(data_class_concat, K=2, loss="Loss2", seed=SEED, Wdist="W1")
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [142.59 254.41]
Obtained loss: 23.360258080327423


$K=4$ (supplementary)

In [6]:
# Selecting K=4 thresholds
best_cutoffs2, min_loss2 = run_de(data_class_concat, K=4, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 81.49 125.04 192.4  274.25]
Obtained loss: 4.911650814731903


In [5]:
# Selecting K=4 thresholds with W_1
best_cutoffs2, min_loss2 = run_de(data_class_concat, K=4, loss="Loss2", seed=SEED, Wdist="W1")
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 81.22 121.16 189.77 266.69]
Obtained loss: 3.6606092702711184


##### $L_1$ loss (supplementary)

In [ ]:
# Selecting K=2 thresholds
best_cutoffs, min_loss = run_de(data_class_concat, K=2, loss="Loss1", seed=SEED)
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [132.34 246.28]
Obtained loss: 309.3412821689561


In [ ]:
# Selecting K=4 thresholds
best_cutoffs, min_loss = run_de(data_class_concat, K=4, loss="Loss1", seed=SEED)
print("Cutoffs:", best_cutoffs[1:-1])
print("Obtained loss:", min_loss)

Cutoffs: [ 76.16 123.44 194.02 279.28]
Obtained loss: 66.98148242930007


### Optimality measures at the traditional thresholds

In [4]:
print("Shah dataset")
print("    L1 at two traditional:", fitness([70, 181], data_class_shah, loss="Loss1"))
print("    L1 at four traditional:", fitness([54, 70, 181, 251], data_class_shah, loss="Loss1"))

# Precompute the Wasserstein distance matrix for L2 loss calculation
data_class_shah.Wdist_matrix()
print("    L2 at two traditional:", fitness([70, 181], data_class_shah, loss="Loss2"))
print("    L2 at four traditional:", fitness([54, 70, 181, 251], data_class_shah, loss="Loss2"))

print("\nBrown dataset")
print("    L1 at two traditional:", fitness([70, 181], data_class_brown, loss="Loss1"))
print("    L1 at four traditional:", fitness([54, 70, 181, 251], data_class_brown, loss="Loss1"))
data_class_brown.Wdist_matrix()
print("    L2 at two traditional:", fitness([70, 181], data_class_brown, loss="Loss2"))
print("    L2 at four traditional:", fitness([54, 70, 181, 251], data_class_brown, loss="Loss2"))

print("\nConcatenated dataset")
data_class_concat.Wdist_matrix()
print("    L2 at two traditional:", fitness([70, 181], data_class_concat, loss="Loss2"))
print("    L2 at four traditional:", fitness([54, 70, 181, 251], data_class_concat, loss="Loss2"))
print("    L1 at two traditional:", fitness([70, 181], data_class_concat, loss="Loss1"))
print("    L1 at four traditional:", fitness([54, 70, 181, 251], data_class_concat, loss="Loss1"))

Shah dataset
    L1 at two traditional: 656.8785795451477
    L1 at four traditional: 655.939314472095
    L2 at two traditional: 42.32659087264856
    L2 at four traditional: 43.67919337267073

Brown dataset
    L1 at two traditional: 1235.9789988634705
    L1 at four traditional: 159.90153344550367
    L2 at two traditional: 142.3521300766216
    L2 at four traditional: 12.2071382439562

Concatenated dataset
    L2 at two traditional: 111.58074625992907
    L2 at four traditional: 144.20941327950624
    L1 at two traditional: 946.428789204309
    L1 at four traditional: 407.92042395879935


### Logistic regression on each TIR proportion

In [5]:
data_concat = pd.concat([grouped_data_shah, grouped_data_brown], ignore_index=True)
data_class_concat = Distribution(data_concat["gl"], ran=(39., 401.), M=200)

In [6]:
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression

np.set_printoptions(precision=5, suppress=True)


def pooled_quantile_cutoffs(data_class, k):
    """Pooled empirical quantiles for a K-threshold naive baseline."""
    pooled = np.concatenate([np.asarray(values, dtype=float) for values in data_class.data])
    probs = np.arange(1, k + 1, dtype=float) / (k + 1)
    return np.quantile(pooled, probs)


def _format_cutoff(value):
    rounded = int(round(float(value)))
    if np.isclose(value, rounded):
        return str(rounded)
    return f"{float(value):.1f}"


def tir_interval_label(lower, upper, idx, total_bins):
    """Readable TIR label for each interval defined by consecutive thresholds."""
    if idx == 0:
        return f"TIR <{_format_cutoff(upper)}"
    if idx == total_bins - 1:
        return f"TIR >={_format_cutoff(lower)}"
    if np.isclose(lower, round(lower)) and np.isclose(upper, round(upper)):
        return f"TIR {int(round(lower))}-{int(round(upper)) - 1}"
    return f"TIR {_format_cutoff(lower)}-{_format_cutoff(upper)}"


def tir_proportions(data_class, cutoffs):
    bins = np.r_[data_class.ran[0], np.asarray(cutoffs, dtype=float), data_class.ran[1]]
    proportions = np.empty((len(data_class.data), len(bins) - 1), dtype=float)
    for row_idx, values in enumerate(data_class.data):
        hist, _ = np.histogram(values, bins=bins, density=True)
        proportions[row_idx] = hist * np.diff(bins)
    return proportions, bins


def build_logistic_regression_table(data_class, threshold_sets, positive_count):
    """
    Compare univariate logistic regressions across threshold sets for one cohort pair.
    The first `positive_count` subjects are treated as class 1.
    """
    target = np.r_[
        np.ones(positive_count, dtype=int),
        np.zeros(len(data_class.data) - positive_count, dtype=int),
    ]

    method_frames = []
    for method_name, cutoffs in threshold_sets.items():
        proportions, bins = tir_proportions(data_class, cutoffs)
        rows = []
        total_bins = len(bins) - 1

        for idx in range(total_bins):
            feature = proportions[:, idx].reshape(-1, 1)
            logreg = LogisticRegression(penalty=None, max_iter=1000)
            logreg.fit(feature, target)
            rows.append(
                {
                    "Range": tir_interval_label(bins[idx], bins[idx + 1], idx, total_bins),
                    "Accuracy (%)": 100 * logreg.score(feature, target),
                    # "Decision boundary": float(-logreg.intercept_[0] / logreg.coef_[0, 0]),
                }
            )

        frame = pd.DataFrame(rows)
        frame.columns = pd.MultiIndex.from_product([[method_name], frame.columns])
        method_frames.append(frame)

    return pd.concat(method_frames, axis=1)


def style_logistic_regression_table(table_df):
    formatters = {}
    for column in table_df.columns:
        if column[1] == "Accuracy (%)":
            formatters[column] = "{:.1f}".format
        elif column[1] == "Decision boundary":
            formatters[column] = "{:.3f}".format
    return table_df.style.hide(axis="index").format(formatters)

In [7]:
combined_l2_k2_logreg_df = build_logistic_regression_table(
    data_class_concat,
    threshold_sets={
        "DE": [150, 258],
        "Consensus": [70, 181],
    },
    positive_count=len(grouped_data_shah),
)

style_logistic_regression_table(combined_l2_k2_logreg_df)

In [8]:
combined_l2_k4_logreg_df = build_logistic_regression_table(
    data_class_concat,
    threshold_sets={
        "DE": [82, 126, 193, 275],
        "Consensus": [54, 70, 181, 251],
    },
    positive_count=len(grouped_data_shah),
)

style_logistic_regression_table(combined_l2_k4_logreg_df)

In [9]:
combined_l1_k4_logreg_df = build_logistic_regression_table(
    data_class_concat,
    threshold_sets={
        "DE": [77, 124, 195, 280],
        "Consensus": [54, 70, 181, 251],
    },
    positive_count=len(grouped_data_shah),
)

style_logistic_regression_table(combined_l1_k4_logreg_df)

## Comparison with Naive Thresholding (supplementary)

A natural naive baseline for distributional data is to use pooled glucose quantiles instead of optimized cutoffs. The cell below compares those pooled-quantile thresholds against the same logistic-regression summaries used above.


In [10]:
combined_naive_thresholds = {
    2: pooled_quantile_cutoffs(data_class_concat, 2),
    4: pooled_quantile_cutoffs(data_class_concat, 4),
}

combined_naive_threshold_summary_df = pd.DataFrame(
    {
        "K": [2, 4],
        "Naive thresholds": [
            ", ".join(_format_cutoff(value) for value in combined_naive_thresholds[2]),
            ", ".join(_format_cutoff(value) for value in combined_naive_thresholds[4]),
        ],
    }
)

combined_l2_k2_with_naive_df = build_logistic_regression_table(
    data_class_concat,
    threshold_sets={
        "Consensus": [70, 181],
        "DE": [150, 258],
        "Naive": combined_naive_thresholds[2],
    },
    positive_count=len(grouped_data_shah),
)

combined_l2_k4_with_naive_df = build_logistic_regression_table(
    data_class_concat,
    threshold_sets={
        "Consensus": [54, 70, 181, 251],
        "DE": [82, 126, 193, 275],
        "Naive": combined_naive_thresholds[4],
    },
    positive_count=len(grouped_data_shah),
)

display(Markdown("#### Pooled-quantile naive thresholds for the combined cohort"))
display(combined_naive_threshold_summary_df)

display(Markdown("#### Combined data, K=2 (L2 thresholds)"))
display(style_logistic_regression_table(combined_l2_k2_with_naive_df))

display(Markdown("#### Combined data, K=4 (L2 thresholds)"))
display(style_logistic_regression_table(combined_l2_k4_with_naive_df))


#### Pooled-quantile naive thresholds for the combined cohort

,K,Naive thresholds
0,2,"123, 173"
1,4,"107, 132, 161, 204"


#### Combined data, K=2 (L2 thresholds)

#### Combined data, K=4 (L2 thresholds)

In [13]:
from pathlib import Path

output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ["Consensus", "DE", "Naive"]


def _latex_range(range_str):
    """Wrap math symbols in Range labels so LaTeX renders them correctly."""
    import re
    range_str = re.sub(r">=", r"$\\geq$", range_str)
    range_str = re.sub(r"<", r"$<$", range_str)
    range_str = range_str.replace("--", "--")  # en-dash is fine
    return range_str


def logreg_table_to_latex(df, k, loss_label="L_2"):
    """Convert a logistic-regression comparison DataFrame to a LaTeX table."""
    methods = [m for m in METHOD_ORDER if m in df.columns.get_level_values(0)]
    n_methods = len(methods)
    col_spec = "lr" * n_methods  # Range + Accuracy for each method

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{Univariate logistic-regression accuracy (\%) on the combined cohort "
        rf"using $K={k}$ thresholds (${loss_label}$ loss). "
        r"Each TIR proportion is used as the sole predictor for classifying individuals "
        r"without diabetes vs.\ with type~1 diabetes.}",
        rf"\label{{tab:combined_logreg_k{k}}}",
        r"\small",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
    ]

    # Method header
    header_parts = []
    for i, method in enumerate(methods):
        col_start = 1 + i * 2
        col_end = col_start + 1
        header_parts.append(rf"\multicolumn{{2}}{{c}}{{{method}}}")
    lines.append(" & ".join(header_parts) + r" \\")

    # Cmidrules
    cmidrules = []
    for i in range(n_methods):
        col_start = 1 + i * 2
        col_end = col_start + 1
        cmidrules.append(rf"\cmidrule(lr){{{col_start}-{col_end}}}")
    lines.append(" ".join(cmidrules))

    # Sub-header
    sub_parts = []
    for method in methods:
        sub_parts.append("Range & Accuracy(\\%)")
    lines.append(" & ".join(sub_parts) + r" \\")
    lines.append(r"\midrule")

    # Data rows — find max row count across methods
    n_rows = max(len(df[m]["Range"].dropna()) for m in methods)
    for row_idx in range(n_rows):
        parts = []
        for method in methods:
            range_col = df[(method, "Range")]
            acc_col = df[(method, "Accuracy (%)")]
            if row_idx < len(range_col):
                range_val = _latex_range(str(range_col.iloc[row_idx]))
                acc_val = acc_col.iloc[row_idx]
                parts.append(f"{range_val} & {acc_val:.1f}")
            else:
                parts.append("& ")
        lines.append(" & ".join(parts) + r" \\")

    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    return "\n".join(lines)


tex_k2 = logreg_table_to_latex(combined_l2_k2_with_naive_df, k=2)
tex_k4 = logreg_table_to_latex(combined_l2_k4_with_naive_df, k=4)

(output_dir / "combined_logreg_k2.tex").write_text(tex_k2, encoding="utf-8")
(output_dir / "combined_logreg_k4.tex").write_text(tex_k4, encoding="utf-8")

print("Exported:")
print(f"  {output_dir / 'combined_logreg_k2.tex'}")
print(f"  {output_dir / 'combined_logreg_k4.tex'}")
print()
print(tex_k2)
print()
print(tex_k4)

Exported:
  results\tables\combined_logreg_k2.tex
  results\tables\combined_logreg_k4.tex

\begin{table}[t]
\centering
\caption{Univariate logistic-regression accuracy (\%) on the combined cohort using $K=2$ thresholds ($L_2$ loss). Each TIR proportion is used as the sole predictor for classifying individuals without diabetes vs.\ with type~1 diabetes.}
\label{tab:combined_logreg_k2}
\small
\begin{tabular}{lrlrlr}
\toprule
\multicolumn{2}{c}{Consensus} & \multicolumn{2}{c}{DE} & \multicolumn{2}{c}{Naive} \\
\cmidrule(lr){1-2} \cmidrule(lr){3-4} \cmidrule(lr){5-6}
Range & Accuracy(\%) & Range & Accuracy(\%) & Range & Accuracy(\%) \\
\midrule
TIR $<$70 & 47.0 & TIR $<$150 & 100.0 & TIR $<$123 & 100.0 \\
TIR 70-180 & 97.9 & TIR 150-257 & 100.0 & TIR 123-172 & 94.3 \\
TIR $\geq$181 & 100.0 & TIR $\geq$258 & 99.1 & TIR $\geq$173 & 100.0 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}[t]
\centering
\caption{Univariate logistic-regression accuracy (\%) on the combined cohort using $K=

## Discriminative Analysis with Wasserstein Regression

For binary discrimination, we modify the WR method: the WR package is used to build the subject-level FPCA score representation of each glucose distribution, which are used as linear predictors in the original WR method, and those scores are then passed to a binomial logistic regression as the second-stage model.


In [4]:
from IPython.display import Markdown, display

from tools.r_tools import ensure_r_packages, setup_r_environment
from tools.downstream_tools import compute_quantile_matrix, fit_wr_binomial_model


def wr_binomial_training_accuracy(glucose_lists, y, probs=None, link="logit"):
    if probs is None:
        probs = np.linspace(0.0, 1.0, 101)

    y = np.asarray(y, dtype=int)
    qf_matrix = compute_quantile_matrix(glucose_lists, probs)
    wr_fit = fit_wr_binomial_model(y, qf_matrix, probs, link=link)

    fitted_prob = np.asarray(wr_fit["fitted_prob"], dtype=float)
    predicted = (fitted_prob >= 0.5).astype(int)
    accuracy = 100 * np.mean(predicted == y)

    summary_df = pd.DataFrame(
        {
            "Metric": ["Training accuracy (%)"],
            "Value": [f"{accuracy:.1f}"],
        }
    ).set_index("Metric")

    return {
        "wr_fit": wr_fit,
        "summary_df": summary_df,
        "predicted": predicted,
        "fitted_prob": fitted_prob,
    }


In [ ]:
setup_r_environment()
combined_wr_probs = np.linspace(0.0, 1.0, 101)
combined_wr_labels = np.r_[np.ones(len(grouped_data_shah), dtype=int), np.zeros(len(grouped_data_brown), dtype=int)]

combined_wr_binomial_report = wr_binomial_training_accuracy(
    glucose_lists=data_concat["gl"],
    y=combined_wr_labels,
    probs=combined_wr_probs,
)

combined_wr_binomial_fit = combined_wr_binomial_report["wr_fit"]
combined_wr_binomial_summary_df = combined_wr_binomial_report["summary_df"]

In [7]:
display(Markdown("#### Combined data: WR-FPCA + logistic regression"))
display(combined_wr_binomial_summary_df)

#### Combined data: WR-FPCA + logistic regression

,Value
Metric,
Training accuracy (%),100.0
